# Research Mate · PubMed Keyword-Aware Hybrid Retrieval

**Demo:** local SQLite cache → PubMed fallback → author keywords + MeSH extraction → keyword-aware **BM25 + TF-IDF** and **MiniLM** → three-way RRF → JSON output.

**Important:** Author keywords and MeSH may be absent from some PubMed records. Retrieval quality remains to be evaluated with labeled queries.


## 1 · Install packages

Run once in your notebook kernel, then **restart the kernel** if prompted. Using `%pip` installs into the active Jupyter environment.

In [1]:
%pip install -q requests pandas numpy rank-bm25 sentence-transformers scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2 · Configuration

For NCBI usage, enter your email address in `NCBI_EMAIL` (strongly recommended; use your own email). Optional API key increases request limits. Do **not** commit credentials to Git. Set `FORCE_PUBMED_REFRESH=True` in the demo if you want to show the fallback even with a populated cache.

**Model download:** The first MiniLM run downloads model weights; subsequent runs reuse the model cache. On a memory-limited laptop, keep `MAX_PAPERS_PER_FETCH` small.

In [2]:
from pathlib import Path
import hashlib
import json
import os
import re
import sqlite3
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from rank_bm25 import BM25Okapi

DB_PATH = Path("researchmate_pubmed.sqlite")
OUTPUT_PATH = Path("researchmate_results.json")
NCBI_EMAIL = os.getenv("NCBI_EMAIL", "")  # Optional: set your own email here
NCBI_API_KEY = os.getenv("NCBI_API_KEY", "")
NCBI_TOOL = "ResearchMatePrototype"
USER_AGENT = f"{NCBI_TOOL}/0.1 ({NCBI_EMAIL or 'local research demo'})"
MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_PAPERS_PER_FETCH = 30
MIN_LOCAL_RELEVANT = 5
# Heuristic for deciding whether local results are adequate (not calibrated).
MIN_COSINE_FOR_LOCAL = 0.35
FORCE_PUBMED_REFRESH = False

print("SQLite cache:", DB_PATH.resolve())

SQLite cache: /Users/ekta/Desktop/Sabudh Internship/ResearchMate/Code/Integration/researchmate_pubmed.sqlite


## 3 · Create or migrate the local paper store

Stores author keywords and MeSH terms as JSON arrays in SQLite. Existing databases are automatically migrated; older records still need re-fetching for their keywords to be populated.


In [3]:
def connect_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("""
        CREATE TABLE IF NOT EXISTS papers (
            pmid TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            abstract TEXT NOT NULL DEFAULT '',
            doi TEXT,
            journal TEXT,
            year TEXT,
            keywords TEXT NOT NULL DEFAULT '[]',
            mesh_terms TEXT NOT NULL DEFAULT '[]',
            source TEXT NOT NULL DEFAULT 'pubmed',
            fetched_at TEXT NOT NULL
        )
    """)
    # Automatically migrate databases created by the previous notebook.
    existing = {row[1] for row in conn.execute("PRAGMA table_info(papers)")}
    for col in ("keywords", "mesh_terms"):
        if col not in existing:
            conn.execute(f"ALTER TABLE papers ADD COLUMN {col} TEXT NOT NULL DEFAULT '[]'")
    conn.commit()
    return conn


def upsert_papers(papers):
    if not papers:
        return 0
    rows = []
    for paper in papers:
        row = dict(paper)
        for field in ("keywords", "mesh_terms"):
            value = row.get(field, [])
            row[field] = json.dumps(value if isinstance(value, list) else json.loads(value or '[]'), ensure_ascii=False)
        rows.append(row)
    with connect_db() as conn:
        conn.executemany("""
            INSERT INTO papers (pmid, title, abstract, doi, journal, year,
                                keywords, mesh_terms, source, fetched_at)
            VALUES (:pmid, :title, :abstract, :doi, :journal, :year,
                    :keywords, :mesh_terms, :source, :fetched_at)
            ON CONFLICT(pmid) DO UPDATE SET
                title=excluded.title, abstract=excluded.abstract,
                doi=excluded.doi, journal=excluded.journal,
                year=excluded.year, keywords=excluded.keywords,
                mesh_terms=excluded.mesh_terms, fetched_at=excluded.fetched_at
        """, rows)
    return len(rows)


def load_papers():
    with connect_db() as conn:
        papers = [dict(r) for r in conn.execute(
            "SELECT * FROM papers WHERE LENGTH(TRIM(abstract)) > 0 ORDER BY pmid"
        )]
    for paper in papers:
        for field in ("keywords", "mesh_terms"):
            paper[field] = json.loads(paper[field] or '[]')
    return papers

connect_db().close()
print("Local papers with abstracts:", len(load_papers()))


Local papers with abstracts: 29


## 4 · PubMed API integration

NCBI ESearch finds PMIDs; EFetch XML provides title, abstract, DOI, **author keywords** (`KeywordList`) and **MeSH descriptors / qualifiers** (`MeshHeadingList`). These fields are optional and may be missing.


In [4]:
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})
_last_request_time = 0.0


def ncbi_get(endpoint, params, timeout=35):
    global _last_request_time
    params = {**params, "tool": NCBI_TOOL}
    if NCBI_EMAIL:
        params["email"] = NCBI_EMAIL
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    # For API-key-free requests, allow >= 0.35s between calls.
    wait = (0.12 if NCBI_API_KEY else 0.35) - (time.monotonic() - _last_request_time)
    if wait > 0:
        time.sleep(wait)
    try:
        response = SESSION.get(f"{BASE_URL}/{endpoint}", params=params, timeout=timeout)
        _last_request_time = time.monotonic()
        response.raise_for_status()
        return response
    except requests.RequestException as exc:
        raise RuntimeError(f"PubMed request failed ({endpoint}): {exc}") from exc


def pubmed_search(query, retmax=30):
    response = ncbi_get("esearch.fcgi", {
        "db": "pubmed", "term": query, "retmode": "json", "retmax": int(retmax),
        "sort": "relevance"
    })
    return response.json().get("esearchresult", {}).get("idlist", [])


def node_text(node):
    return " ".join(" ".join(node.itertext()).split()) if node is not None else ""


def parse_pubmed_xml(xml_data):
    root = ET.fromstring(xml_data)
    items = []
    now = datetime.now(timezone.utc).isoformat()
    for rec in root.findall(".//PubmedArticle"):
        pmid = node_text(rec.find("./MedlineCitation/PMID"))
        article = rec.find("./MedlineCitation/Article")
        if not pmid or article is None:
            continue
        title = node_text(article.find("ArticleTitle"))
        abstract_node = article.find("Abstract")
        parts = []
        if abstract_node is not None:
            for sec in abstract_node.findall("AbstractText"):
                label = sec.attrib.get("Label", "")
                body = node_text(sec)
                if body:
                    parts.append(f"{label}: {body}" if label else body)
        abstract = " ".join(parts).strip()
        if not abstract:  # No abstract available to rank
            continue
        doi = next((node_text(x) for x in rec.findall("./PubmedData/ArticleIdList/ArticleId")
                    if x.attrib.get("IdType") == "doi"), None)
        if not doi:
            doi = next((node_text(x) for x in article.findall("ELocationID")
                        if x.attrib.get("EIdType") == "doi"), None)
        # PubMed author keywords may live in one or more KeywordList elements.
        keywords = list(dict.fromkeys(
            text for item in rec.findall("./MedlineCitation/KeywordList/Keyword")
            if (text := node_text(item))
        ))
        # MeSH descriptors are controlled vocabulary; qualifiers give added specificity.
        mesh_terms = []
        for heading in rec.findall("./MedlineCitation/MeshHeadingList/MeshHeading"):
            descriptor = node_text(heading.find("DescriptorName"))
            if descriptor:
                mesh_terms.append(descriptor)
            mesh_terms.extend(node_text(q) for q in heading.findall("QualifierName") if node_text(q))
        mesh_terms = list(dict.fromkeys(mesh_terms))
        journal = node_text(article.find("./Journal/Title"))
        year = node_text(article.find("./Journal/JournalIssue/PubDate/Year"))
        if not year:
            medline_date = node_text(article.find("./Journal/JournalIssue/PubDate/MedlineDate"))
            match = re.search(r"\b(?:19|20)\d{2}\b", medline_date)
            year = match.group(0) if match else ""
        items.append({"pmid": pmid, "title": title, "abstract": abstract,
                      "doi": doi, "journal": journal, "year": year,
                      "keywords": keywords, "mesh_terms": mesh_terms,
                      "source": "pubmed", "fetched_at": now})
    return items


def pubmed_fetch(pmids, batch_size=100):
    papers = []
    for i in range(0, len(pmids), batch_size):
        chunk = pmids[i:i + batch_size]
        if not chunk:
            continue
        response = ncbi_get("efetch.fcgi", {
            "db": "pubmed", "id": ",".join(chunk), "retmode": "xml"
        }, timeout=60)
        papers.extend(parse_pubmed_xml(response.content))
    return papers

print("PubMed functions ready. No request sent yet.")

PubMed functions ready. No request sent yet.


## 5 · Keyword-aware BM25 + TF-IDF, MiniLM, and RRF

Both **BM25** and **TF-IDF** search `title (2×) + abstract (1×) + author keywords (2×) + MeSH terms (2×)` using a simple repetition-based weighting heuristic. MiniLM remains title+abstract to preserve a comparable semantic baseline. RRF combines three ranked lists without mixing incompatible raw score scales. Adjust weighting after relevance evaluation.


In [5]:
from functools import lru_cache
from sklearn.feature_extraction.text import TfidfVectorizer

@lru_cache(maxsize=1)
def get_encoder():
    from sentence_transformers import SentenceTransformer
    print("Loading MiniLM (first run may download model weights)...")
    return SentenceTransformer(MINILM_MODEL, device="cpu")


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def lexical_text(paper):
    """Consistent field-weighting for both TF-IDF and BM25.

    Keywords and MeSH terms repeated twice, title repeated twice, abstract once.
    These are simple tunable heuristics, not validated relevance weights.
    """
    title = paper.get("title", "") or ""
    abstract = paper.get("abstract", "") or ""
    keywords = paper.get("keywords", []) or []
    mesh_terms = paper.get("mesh_terms", []) or []
    if isinstance(keywords, str):
        keywords = json.loads(keywords)
    if isinstance(mesh_terms, str):
        mesh_terms = json.loads(mesh_terms)
    metadata = " ".join([*keywords, *mesh_terms])
    return " ".join([title, title, abstract, metadata, metadata]).strip()


def rank_bm25(query, papers):
    if not papers:
        return []
    corpus = [tokenize(lexical_text(p)) for p in papers]
    scores = BM25Okapi(corpus).get_scores(tokenize(query))
    return sorted(zip(papers, map(float, scores)), key=lambda x: x[1], reverse=True)


def rank_tfidf(query, papers):
    if not papers:
        return []
    texts = [lexical_text(p) for p in papers]
    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
    matrix = vectorizer.fit_transform(texts)
    query_vec = vectorizer.transform([query])
    # TF-IDF vectors are L2-normalized by default; dot product is cosine similarity.
    scores = (matrix @ query_vec.T).toarray().ravel()
    return sorted(zip(papers, map(float, scores)), key=lambda x: x[1], reverse=True)


def rank_minilm(query, papers):
    if not papers:
        return []
    encoder = get_encoder()
    # Keep semantic baseline unchanged: title + abstract, no keyword repetition.
    texts = [p["title"] + ". " + p["abstract"] for p in papers]
    doc_vectors = encoder.encode(texts, batch_size=16, normalize_embeddings=True,
                                 show_progress_bar=False, convert_to_numpy=True)
    query_vector = encoder.encode([query], normalize_embeddings=True,
                                  convert_to_numpy=True)[0]
    scores = doc_vectors @ query_vector
    return sorted(zip(papers, map(float, scores)), key=lambda x: x[1], reverse=True)


def reciprocal_rank_fusion(*rankings, rrf_k=60):
    fused, paper_by_id = {}, {}
    for ranking in rankings:
        for position, (paper, _) in enumerate(ranking, start=1):
            pmid = paper["pmid"]
            paper_by_id[pmid] = paper
            fused[pmid] = fused.get(pmid, 0.0) + 1.0 / (rrf_k + position)
    ordered = sorted(fused, key=lambda pmid: (-fused[pmid], pmid))
    return [(paper_by_id[pmid], float(fused[pmid])) for pmid in ordered]


def run_hybrid(query, papers, top_k=5):
    if not papers:
        return [], {}, {}
    bm25 = rank_bm25(query, papers)
    tfidf = rank_tfidf(query, papers)
    semantic = rank_minilm(query, papers)
    fused = reciprocal_rank_fusion(bm25, tfidf, semantic)
    scores = {p["pmid"]: {} for p in papers}
    for name, ranking, score_name in (
        ("bm25", bm25, "bm25_score"),
        ("tfidf", tfidf, "tfidf_score"),
        ("minilm", semantic, "cosine_similarity"),
    ):
        for pos, (paper, score) in enumerate(ranking, start=1):
            scores[paper["pmid"]][f"{name}_rank"] = pos
            scores[paper["pmid"]][score_name] = round(score, 4)
    return fused[:top_k], scores, {"bm25": bm25, "tfidf": tfidf, "minilm": semantic}

print("BM25, TF-IDF, MiniLM and three-way RRF ready.")


BM25, TF-IDF, MiniLM and three-way RRF ready.


## 6 · End-to-end retrieval function

Local-first cache with PubMed fallback, returning parsed `keywords`, `mesh_terms`, BM25, TF-IDF, MiniLM scores/ranks, and hybrid RRF ranks. The current local sufficiency rule uses a heuristic MiniLM threshold, not validated relevance.


In [6]:
def retrieve(query, domain="biology", top_k=5, pubmed_retmax=30,
             force_refresh=False, verbose=True):
    if domain.lower() != "biology":
        raise ValueError("This PubMed prototype currently supports domain='biology' only.")
    if not query.strip():
        raise ValueError("Query cannot be empty.")
    started = time.perf_counter()
    local = load_papers()
    initial_ids = {p["pmid"] for p in local}
    local_relevant = 0
    if local and not force_refresh:
        candidate_rank = rank_minilm(query, local)
        local_relevant = sum(s >= MIN_COSINE_FOR_LOCAL for _, s in candidate_rank)

    needs_api = force_refresh or local_relevant < MIN_LOCAL_RELEVANT
    api_status = "not_needed"
    fetched = []
    if needs_api:
        try:
            ids = pubmed_search(query, retmax=pubmed_retmax)
            fetched = pubmed_fetch(ids)
            upsert_papers(fetched)
            api_status = "success"
        except (RuntimeError, ValueError, ET.ParseError) as exc:
            api_status = f"error: {exc}"
            if not local:
                raise RuntimeError("PubMed unavailable and the local cache is empty.") from exc
            if verbose:
                print("PubMed unavailable; using cached papers:", exc)

    corpus = load_papers()
    fused, scores, rankings = run_hybrid(query, corpus, top_k=top_k)
    results = []
    for rank, (paper, rrf_score) in enumerate(fused, start=1):
        pmid = paper["pmid"]
        results.append({
            "rank": rank,
            "paper_id": pmid,
            "pmid": pmid,
            "title": paper["title"],
            "abstract": paper["abstract"],
            "text": paper["abstract"],  # future integration contract
            "doi": paper["doi"],
            "keywords": paper["keywords"],
            "mesh_terms": paper["mesh_terms"],
            "year": paper["year"],
            "journal": paper["journal"],
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "source": paper["source"],
            "retrieval_origin": "pubmed_api" if pmid not in initial_ids else "local_cache",
            "score": round(rrf_score, 6),  # RRF score, not a probability
            **scores[pmid]
        })
    payload = {
        "query": query,
        "domain": domain,
        "method": "BM25+TF-IDF+MiniLM+RRF",
        "api_status": api_status,
        "local_relevant_before_fetch": local_relevant,
        "new_papers_fetched_with_abstracts": len(fetched),
        "cached_papers_with_abstracts": len(corpus),
        "elapsed_seconds": round(time.perf_counter() - started, 2),
        "results": results
    }
    if verbose:
        print(f"Retrieved {len(results)} results | API: {api_status} | "
              f"local corpus: {len(corpus)} | {payload['elapsed_seconds']}s")
    return payload

print("retrieve(query, domain='biology', top_k=5) is ready.")

retrieve(query, domain='biology', top_k=5) is ready.


## 7 · Live demonstration

**Existing SQLite users:** set `FORCE_PUBMED_REFRESH = True` in configuration once for a demonstration that refetches the selected query's PubMed records and populates their keyword fields. This **does not backfill every older cached record**. To backfill all, fetch the stored PMIDs in batches using `pubmed_fetch` and call `upsert_papers`. Return `FORCE_PUBMED_REFRESH` to False for local-first demonstration.


In [7]:
QUERY = "How does temperature influence adaptation in Drosophila?"

# If your SQLite cache predates the keyword update, set this True once
# to refresh existing PubMed records and populate keyword/MeSH fields.
response = retrieve(
    QUERY, domain="biology", top_k=5,
    pubmed_retmax=MAX_PAPERS_PER_FETCH,
    force_refresh=FORCE_PUBMED_REFRESH
)

columns = ["rank", "title", "year", "pmid", "keywords", "mesh_terms",
           "bm25_rank", "tfidf_rank", "minilm_rank", "score", "retrieval_origin"]
display(pd.DataFrame(response["results"], columns=columns))


/Users/ekta/Desktop/Sabudh Internship/ResearchMate/Code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1120 09:22:22.601000 12779 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading MiniLM (first run may download model weights)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4828.42it/s]


Retrieved 5 results | API: not_needed | local corpus: 29 | 12.09s


,rank,title,year,pmid,keywords,mesh_terms,bm25_rank,tfidf_rank,minilm_rank,score,retrieval_origin
0,1,Comparative analysis of temperature preference...,2022,35879333,[],[],1,3,2,0.048395,local_cache
1,2,Population History Shapes Responses to Differe...,2023,37374116,[],[],2,1,3,0.048395,local_cache
2,3,Temperature-driven gene expression evolution i...,2023,37455661,[],[],4,2,6,0.046906,local_cache
3,4,Temperature-mediated dynamics: Unravelling the...,2024,39263086,[],[],10,6,1,0.045831,local_cache
4,5,The effect of environmental temperature on olf...,2009,19559705,[],[],14,5,10,0.043184,local_cache


## 8 · Inspect rankings and export standardized JSON

This gives the frontend team a concrete data contract, while keeping the model-scoring details available for evaluation. All results are linked to PMID records for traceable citation.

In [8]:
OUTPUT_PATH.write_text(json.dumps(response, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", OUTPUT_PATH.resolve())
print("API status:", response["api_status"])
print("Top 1:", response["results"][0]["title"] if response["results"] else "No matches")
print("Integration payload keys:", list(response.keys()))
print("Result fields:", list(response["results"][0].keys()) if response["results"] else [])

Saved: /Users/ekta/Desktop/Sabudh Internship/ResearchMate/Code/Integration/researchmate_results.json
API status: not_needed
Top 1: Comparative analysis of temperature preference behavior and effects of temperature on daily behavior in 11 Drosophila species.
Integration payload keys: ['query', 'domain', 'method', 'api_status', 'local_relevant_before_fetch', 'new_papers_fetched_with_abstracts', 'cached_papers_with_abstracts', 'elapsed_seconds', 'results']
Result fields: ['rank', 'paper_id', 'pmid', 'title', 'abstract', 'text', 'doi', 'keywords', 'mesh_terms', 'year', 'journal', 'url', 'source', 'retrieval_origin', 'score', 'bm25_rank', 'bm25_score', 'tfidf_rank', 'tfidf_score', 'minilm_rank', 'cosine_similarity']


## 9 · Side-by-side BM25, TF-IDF, MiniLM and hybrid rankings

Show differences but do not conclude accuracy improvements without labeled relevance judgments.


In [9]:
papers = load_papers()
bm25_order = rank_bm25(QUERY, papers)
tfidf_order = rank_tfidf(QUERY, papers)
minilm_order = rank_minilm(QUERY, papers)
hybrid_order = reciprocal_rank_fusion(bm25_order, tfidf_order, minilm_order)
rows = []
for i in range(min(5, len(papers))):
    rows.append({
        "position": i + 1,
        "BM25": bm25_order[i][0]["title"][:65],
        "TF-IDF": tfidf_order[i][0]["title"][:65],
        "MiniLM": minilm_order[i][0]["title"][:65],
        "Hybrid RRF": hybrid_order[i][0]["title"][:65]
    })
display(pd.DataFrame(rows))


,position,BM25,TF-IDF,MiniLM,Hybrid RRF
0,1,Comparative analysis of temperature preference...,Population History Shapes Responses to Differe...,Temperature-mediated dynamics: Unravelling the...,Comparative analysis of temperature preference...
1,2,Population History Shapes Responses to Differe...,Temperature-driven gene expression evolution i...,Comparative analysis of temperature preference...,Population History Shapes Responses to Differe...
2,3,The behavior of adult Drosophila in the wild.,Comparative analysis of temperature preference...,Population History Shapes Responses to Differe...,Temperature-driven gene expression evolution i...
3,4,Temperature-driven gene expression evolution i...,The behavior of adult Drosophila in the wild.,Thermosensory perception regulates speed of mo...,Temperature-mediated dynamics: Unravelling the...
4,5,Microbiome composition shapes temperature tole...,The effect of environmental temperature on olf...,Mating success at elevated temperature is asso...,The effect of environmental temperature on olf...


## 10 · Optional: small regression test (no PubMed call)

Use this to check ranking behavior after you change the retrieval pipeline. This tests consistency and output shapes, **not** relevance quality.

In [10]:
def smoke_test():
    samples = [
        {"pmid": "1", "title": "Thermal adaptation in fruit flies",
         "abstract": "Temperature drives adaptation in Drosophila populations.",
         "keywords": ["thermal adaptation"], "mesh_terms": ["Drosophila"]},
        {"pmid": "2", "title": "Machine learning overview",
         "abstract": "Neural networks predict images and labels.",
         "keywords": ["deep learning"], "mesh_terms": []},
        {"pmid": "3", "title": "Heat tolerance in insects",
         "abstract": "Insects evolve heat resistance in warmer environments.",
         "keywords": ["temperature"], "mesh_terms": ["Insects"]}
    ]
    bm25 = rank_bm25("thermal adaptation Drosophila", samples)
    tfidf = rank_tfidf("thermal adaptation Drosophila", samples)
    assert bm25[0][0]["pmid"] == "1"
    assert tfidf[0][0]["pmid"] == "1"
    fused, scores, ranks = run_hybrid("thermal adaptation Drosophila", samples, top_k=2)
    assert len(fused) == 2
    assert len({p["pmid"] for p, _ in fused}) == 2
    assert set(scores) == {"1", "2", "3"}
    assert all(score > 0 for _, score in fused)
    assert all("tfidf_score" in s for s in scores.values())
    print("Smoke test passed.")

smoke_test()


Smoke test passed.


## Talking points for the meeting

- **Independently runnable:** PubMed + SQLite, no frontend or team Elasticsearch dependency.
- **New metadata:** author keywords and MeSH subject headings are fetched, stored and exported.
- **Lexical search:** TF-IDF and BM25 explicitly include keywords and MeSH with tunable field weights.
- **Semantic baseline:** MiniLM title+abstract remains unchanged.
- **Hybrid:** RRF now combines BM25, TF-IDF and MiniLM ranks.
- **Next:** label relevant results and benchmark inclusion/exclusion of keywords; then connect to shared database/API.
